# rl18_report

> Phase 18, Part B (`specs/multiagent/phase18/`): the analysis of the clone-RL runs on the colony and plain boards. Per run its mean over evaluation boards; H18-B1 (exact permutation test, colony-trained against plain-trained runs), H18-B2 (emergent division of labour: specialization index minus its shuffled null, sign-flip over runs), H18-B3 (gain over the warm start at N = 32), Holm over the three; H18-B4 (goal drift against the warm start, descriptive); a report and a figure; `syntheticRows18B` for tests.

In [ ]:
#| default_exp rl18_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Run values and tests

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm
from explRL.rl16_report import permTest, signFlip
from explRL.rl18_grid import RL18_PLAN, RL18_RUNS

NAN = float('nan')


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def runValue(rows, model, test, metric):
    "Mean of metric over a model's evaluation boards of one test (level '<model>|<test>')."
    v = [_f(r[metric]) for r in rows if r['level'] == f'{model}|{test}']
    v = [x for x in v if np.isfinite(x)]
    return float(np.mean(v)) if v else NAN


def runNames(arm, runs=RL18_RUNS):
    return [f'{a}_s{s}' for a, s in runs if a == arm]


def _boot(x, nboot=2000, seed=0):
    x = np.array([v for v in x if np.isfinite(v)], float)
    if not len(x):
        return [NAN, NAN]
    m = x[np.random.default_rng(seed).integers(0, len(x), (nboot, len(x)))].mean(1)
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]


def rl18Summaries(rows, plan=RL18_PLAN, nboot=2000):
    "H18-B1 to B3 with Holm, H18-B4, and per model the main measures at each N (colony and plain boards)."
    C, P = runNames('c18team'), runNames('p18team')
    rv = lambda ms, test, k: [runValue(rows, m, test, k) for m in ms]
    b1 = permTest(rv(C, 'colonyn8liar', 'honestPooled'), rv(P, 'plainn8liar', 'honestPooled'))
    spec = [np.nanmean([runValue(rows, m, t, 'specIndex') - runValue(rows, m, t, 'specNull') for t in ('colonyn8', 'colonyn8liar')]) for m in C]
    b2 = signFlip(spec)
    warm32 = runValue(rows, 'warm', 'colonyn32liar', 'honestPooled')
    g32 = [v - warm32 for v in rv(C, 'colonyn32liar', 'honestPooled')]
    b3 = signFlip(g32)
    hyp = dict(H18_B1=dict(estimate=b1['diff'], p=b1['p'], n=[len(C), len(P)]),
               H18_B2=dict(estimate=b2['mean'], p=b2['p'], n=len(spec), ci=_boot(spec, nboot)),
               H18_B3=dict(estimate=b3['mean'], p=b3['p'], n=len(g32), ci=_boot(g32, nboot)))
    for h, a in zip(plan['holm'], holm([hyp[h]['p'] for h in plan['holm']])):
        hyp[h].update(pHolm=float(a), supported=bool(a < plan['alpha'] and hyp[h]['estimate'] > 0))
    drift = {}
    for arm, runs, test in (('c18team', C, 'colonyn8liar'), ('p18team', P, 'plainn8liar')):
        for k in ('freeRiding', 'misreport'):
            d = [v - runValue(rows, 'warm', test, k) for v in rv(runs, test, k)]
            drift[f'{arm}|{k}'] = dict(estimate=float(np.nanmean(d)) if d else NAN, ci=_boot(d, nboot))
    hyp['H18_B4'] = dict(drift=drift, secondary=True)
    curves = {m: {t: runValue(rows, m, t, 'honestPooled') for t in sorted({r['level'].split('|', 1)[1] for r in rows})}
              for m in ['warm'] + C + P}
    return dict(hyp=hyp, curves=curves, spec=dict(zip(C, spec)), gain32=dict(zip(C, g32)))

## Report and figure

In [ ]:
#| export
def _fmt(x):
    return 'n/a' if x is None or not np.isfinite(x) else f'{x:.3f}'


def writeRL18(path, S, meta):
    "report.md: H18-B1 to B3 (estimate, exact p, Holm), H18-B4 (goal drift), per-run values."
    L = ['# Phase 18 Part B: clones trained with team reward on the colony board (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| hypothesis | estimate | p (one-sided, exact) | Holm | supported |', '|---|---|---|---|---|']
    for h in ('H18_B1', 'H18_B2', 'H18_B3'):
        t = S['hyp'][h]
        L.append(f"| {h} | {_fmt(t['estimate'])} | {_fmt(t['p'])} | {_fmt(t['pHolm'])} | {t['supported']} |")
    L += ['', '## H18-B4: goal drift against the warm start (N 8, with liars)', '', '| arm and measure | change | 95% interval |', '|---|---|---|']
    L += [f"| {k} | {_fmt(v['estimate'])} | [{_fmt(v['ci'][0])}, {_fmt(v['ci'][1])}] |" for k, v in S['hyp']['H18_B4']['drift'].items()]
    L += ['', '## Per run', '', '| run | specialization minus null | gain at N 32 |', '|---|---|---|']
    L += [f"| {m} | {_fmt(S['spec'][m])} | {_fmt(S['gain32'][m])} |" for m in S['spec']]
    open(path, 'w').write('\n'.join(L) + '\n')


def figRL18(out, S):
    "rl18.png: honest pooled share with liars against N on the colony board, the warm start and each run."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.8))
    for m, d in S['curves'].items():
        Ns = sorted(int(t[7:-4]) for t in d if t.startswith('colonyn') and t.endswith('liar'))
        if Ns:
            ax.plot(Ns, [d[f'colonyn{N}liar'] for N in Ns], 'k-o' if m == 'warm' else '-', lw=2 if m == 'warm' else 0.8,
                    color=None if m == 'warm' else ('#1f6391' if m.startswith('c18') else '#c0392b'), label=m if m == 'warm' else None)
    ax.set(xscale='log', xlabel='team size', ylabel='honest pooled share (liars)', title='Colony board: trained clones (blue: colony-trained)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{out}/rl18.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Synthetic rows

In [ ]:
#| export
def syntheticRows18B(gain=0.05, spec=1.0, seed=0, boards=4):
    "Evaluation rows for the warm start and the 13 runs with known effects (c18team: gain over warm and over p18team, specialization)."
    rng, rows = np.random.default_rng(seed), []
    models = ['warm'] + [f'{a}_s{s}' for a, s in RL18_RUNS]
    for m in models:
        g = gain if m.startswith('c18') else 0.0
        for b in ('colony', 'plain'):
            for N in (4, 8, 16, 32):
                for liar in ('', 'liar'):
                    for _ in range(boards):
                        e = rng.normal(0, 0.01)
                        rows.append(dict(level=f'{m}|{b}n{N}{liar}', honestPooled=0.3 + g + e, specIndex=1.0 + (spec if m.startswith('c18') else 0) + e,
                                         specNull=1.0, freeRiding=0.1 + (0.05 if m != 'warm' else 0), misreport=1.0))
    return rows

## Tests

In [ ]:
from explRL.rl18_report import *
S = rl18Summaries(syntheticRows18B(), nboot=200)
h = S['hyp']
assert abs(h['H18_B1']['estimate'] - 0.05) < 0.01 and h['H18_B1']['p'] == 1 / 1287
assert abs(h['H18_B2']['estimate'] - 1.0) < 0.02 and h['H18_B2']['p'] == 1 / 256
assert abs(h['H18_B3']['estimate'] - 0.05) < 0.01 and all(h[k]['supported'] for k in ('H18_B1', 'H18_B2', 'H18_B3'))
assert abs(h['H18_B4']['drift']['c18team|freeRiding']['estimate'] - 0.05) < 1e-9
S0 = rl18Summaries(syntheticRows18B(gain=0.0, spec=0.0), nboot=200)
assert not any(S0['hyp'][k]['supported'] for k in ('H18_B1', 'H18_B2', 'H18_B3'))
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    writeRL18(f'{d}/report.md', S, dict(mode='test'))
    figRL18(d, S)
    assert os.path.getsize(f'{d}/rl18.png') > 0 and 'H18_B2' in open(f'{d}/report.md').read()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()